# Word Embeddings: One-Hot Encoding and Word2Vec

**Runs on:** Google Colab (free CPU)

### What you will learn
1. Why computers need numbers instead of words
2. One-Hot Encoding and its main weakness
3. Training a Word2Vec model (CBOW vs Skip-gram)
4. Finding similar words and measuring word-to-word similarity
5. Visualising word vectors in 2D

## 1. One-Hot Encoding

Each word becomes a vector with a single `1` and the rest `0`. The vector length equals the vocabulary size.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

words = [["king"], ["queen"], ["banana"], ["mango"], ["car"]]

encoder = OneHotEncoder(sparse_output=False)
one_hot = encoder.fit_transform(words)

vocab = encoder.categories_[0]
print("Vocabulary (alphabetical):", list(vocab))
pd.DataFrame(one_hot, index=[w[0] for w in words], columns=vocab)

### Weakness of One-Hot Vectors
Every pair of different words is exactly the same distance apart, so "banana" is no closer to "mango" than to "car".
Also the vectors grow with the vocabulary size (a 50,000-word vocabulary means 50,000 dimensions).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(one_hot)
pd.DataFrame(sim, index=vocab, columns=vocab).round(2)

All off-diagonal values are `0.0`, so one-hot vectors carry **no notion of meaning**. Dense embeddings fix this.

## 2. Word2Vec (Dense Embeddings)

Word2Vec learns a short dense vector for each word by looking at the **words around it**. Words used in similar contexts end up with similar vectors.

In [ ]:
!pip install -q gensim

In [ ]:
import random
from gensim.models import Word2Vec

base_sentences = [
    ["cat", "drinks", "milk"],      ["dog", "drinks", "water"],
    ["cat", "chases", "mouse"],     ["dog", "chases", "ball"],
    ["cat", "is", "a", "pet"],      ["dog", "is", "a", "pet"],
    ["mango", "is", "a", "sweet", "fruit"],  ["apple", "is", "a", "sweet", "fruit"],
    ["mango", "grows", "on", "trees"],       ["apple", "grows", "on", "trees"],
    ["car", "has", "four", "wheels"],        ["bus", "has", "four", "wheels"],
    ["car", "runs", "on", "road"],           ["bus", "runs", "on", "road"],
]

# A tiny corpus is too small to learn from, so repeat and shuffle it
random.seed(42)
corpus = base_sentences * 60
random.shuffle(corpus)
print("Total training sentences:", len(corpus))

In [ ]:
# sg=0 -> CBOW (predict a word from its context)
cbow = Word2Vec(corpus, vector_size=25, window=2, min_count=1, sg=0, epochs=40, workers=1, seed=42)

# sg=1 -> Skip-gram (predict the context from a word)
skip = Word2Vec(corpus, vector_size=25, window=2, min_count=1, sg=1, epochs=40, workers=1, seed=42)

print("Vocabulary size:", len(cbow.wv))
print("Vector for 'cat' (first 8 values):", cbow.wv["cat"][:8].round(3))
print("Vector shape:", cbow.wv["cat"].shape)

### Most similar words

In [ ]:
for w in ["cat", "mango", "car"]:
    print(f"CBOW      most similar to '{w}':", [(x, round(s, 2)) for x, s in cbow.wv.most_similar(w, topn=3)])
    print(f"Skip-gram most similar to '{w}':", [(x, round(s, 2)) for x, s in skip.wv.most_similar(w, topn=3)])
    print()

### Pairwise similarity

In [ ]:
pairs = [("cat", "dog"), ("mango", "apple"), ("car", "bus"), ("cat", "car"), ("dog", "apple")]
rows = [(a, b, round(float(skip.wv.similarity(a, b)), 3)) for a, b in pairs]
pd.DataFrame(rows, columns=["word 1", "word 2", "cosine similarity (skip-gram)"])

Words from the same group (animals, fruits, vehicles) should score higher than words from different groups.

## 3. Visualising the Word Vectors

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

labels = list(skip.wv.index_to_key)
vectors = np.array([skip.wv[w] for w in labels])
coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1])
for i, w in enumerate(labels):
    plt.annotate(w, (coords[i, 0], coords[i, 1]), fontsize=10)
plt.title("Word2Vec vectors reduced to 2D (PCA)")
plt.grid(True)
plt.show()

## 4. Practice Exercise: Effect of Window Size
Train models with `window=1` and `window=4` and compare `similarity('cat', 'dog')`.

In [ ]:
for win in [1, 2, 4]:
    m = Word2Vec(corpus, vector_size=25, window=win, min_count=1, sg=1, epochs=40, workers=1, seed=42)
    print(f"window={win}  similarity(cat, dog) = {m.wv.similarity('cat', 'dog'):.3f}   similarity(cat, car) = {m.wv.similarity('cat', 'car'):.3f}")

## Key Takeaways

| Concept | Meaning |
|---|---|
| One-Hot Encoding | Sparse vector with a single 1, no meaning, size grows with vocabulary |
| Word2Vec | Learns dense vectors from word context |
| CBOW | Predicts a word from its surrounding words (fast) |
| Skip-gram | Predicts surrounding words from a word (better for rare words) |
| `vector_size` | Number of dimensions of each word vector |
| `window` | How many neighbouring words count as context |
| Limitation | One vector per word, so it cannot tell apart different meanings of the same word |